In [1]:
import json

files = [
    "processed/mr_en/train.jsonl",
    "processed/mr_en/val.jsonl",
    "processed/mr_en/test.jsonl"
]

for file in files:
    count = 0

    with open(file, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue

            data = json.loads(line)

            assert "prompt" in data, f"{file} line {line_number}: missing prompt"
            assert "completion" in data, f"{file} line {line_number}: missing completion"

            assert len(data["prompt"]) == 2, \
                f"{file} line {line_number}: prompt should have 2 messages"

            assert data["prompt"][0]["role"] == "system"
            assert data["prompt"][1]["role"] == "user"

            assert len(data["completion"]) == 1
            assert data["completion"][0]["role"] == "assistant"

            # Check that assistant output is valid JSON
            output = json.loads(data["completion"][0]["content"])

            assert output["language"] == "mr-en"
            assert output["sentiment"] in ["positive", "negative", "neutral"]
            assert isinstance(output["hate"], bool)
            assert isinstance(output["profanity"], bool)

            count += 1

    print(f"{file}: {count} examples")

print("\nDataset verification completed successfully.")

processed/mr_en/train.jsonl: 8811 examples
processed/mr_en/val.jsonl: 1535 examples
processed/mr_en/test.jsonl: 1622 examples

Dataset verification completed successfully.


In [2]:
import json


def load_tweets(path):
    tweets = set()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            data = json.loads(line)

            user_message = data["prompt"][1]["content"]

            # Extract everything after "Text: "
            tweet = user_message.split("Text:", 1)[1].strip()

            tweets.add(tweet)

    return tweets


train = load_tweets("processed/mr_en/train.jsonl")
val = load_tweets("processed/mr_en/val.jsonl")
test = load_tweets("processed/mr_en/test.jsonl")


print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

print("\nTrain ∩ Validation:", len(train & val))
print("Train ∩ Test:", len(train & test))
print("Validation ∩ Test:", len(val & test))

Train: 8811
Validation: 1535
Test: 1622

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [3]:
import json
from collections import Counter


def load_labels(path):
    labels = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            data = json.loads(line)

            output = json.loads(
                data["completion"][0]["content"]
            )

            labels.append(output)

    return labels


for split in ["train", "val", "test"]:

    labels = load_labels(f"processed/mr_en/{split}.jsonl")

    sentiment = Counter(x["sentiment"] for x in labels)
    hate = Counter(x["hate"] for x in labels)
    profanity = Counter(x["profanity"] for x in labels)

    print(f"\n===== {split.upper()} =====")

    print("\nSentiment:")
    print(sentiment)

    print("\nHate:")
    print(hate)

    print("\nProfanity:")
    print(profanity)


===== TRAIN =====

Sentiment:
Counter({'positive': 2997, 'negative': 2909, 'neutral': 2905})

Hate:
Counter({False: 8034, True: 777})

Profanity:
Counter({False: 8625, True: 186})

===== VAL =====

Sentiment:
Counter({'negative': 605, 'positive': 468, 'neutral': 462})

Hate:
Counter({False: 1250, True: 285})

Profanity:
Counter({False: 1455, True: 80})

===== TEST =====

Sentiment:
Counter({'negative': 663, 'positive': 491, 'neutral': 468})

Hate:
Counter({False: 1305, True: 317})

Profanity:
Counter({False: 1545, True: 77})


In [4]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

PyTorch: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA version: 11.8
GPU memory: 4.0 GB
